In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("postgres_to_parquet_silver") \
    .master("spark://spark-master:7077") \
    .config("spark.default.parallelism", "4") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10m") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/08 00:28:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
STORAGE_PROTOCOL = "s3a://"
BUCKET_NAME = "end-to-end-streaming-data-platform-bronze"
SOURCE_SUSTEM = "mongo"
FOLDER_NAME = "ingestion_data"
execution_date = "2026-09-06-Sep"
TABLE_NAME = "videos"

MONGO_BRONZE_PATH = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{SOURCE_SUSTEM}/{FOLDER_NAME}={execution_date}/{TABLE_NAME}.parquet"

In [3]:
df = spark.read.parquet(MONGO_BRONZE_PATH)

26/09/07 12:47:19 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

In [2]:
STORAGE_PROTOCOL = "s3a://"
BUCKET_NAME = "end-to-end-streaming-data-platform-silver"
SOURCE_SUSTEM = "mongo"
TABLE_NAME = "videos_silver"

MONGO_SILVER_PATH = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{SOURCE_SUSTEM}/{TABLE_NAME}.parquet"

In [3]:
df_mongo_silver  = spark.read.parquet(MONGO_SILVER_PATH)

26/09/08 00:28:52 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

In [6]:


print("=" * 70)
print("MONGODB SILVER VALIDATION")
print("=" * 70)


# ============================================================
# 1. Record count preserved
# ============================================================

assert df_mongo_silver.count() == df.count()

print("PASS — Record count preserved.")


# ============================================================
# 2. Primary ID uniqueness
# ============================================================

duplicate_video_ids = (
    df_mongo_silver
    .groupBy("video_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_video_ids == 0

print("PASS — video_id uniqueness preserved.")


# ============================================================
# 3. Missing title preserved and flagged
# ============================================================

missing_title_count = (
    df_mongo_silver
    .filter(F.col("title").isNull())
    .count()
)

flagged_missing_title_count = (
    df_mongo_silver
    .filter(F.col("is_missing_title") == True)
    .count()
)

assert missing_title_count == flagged_missing_title_count

print("PASS — Missing titles preserved and flagged.")


# ============================================================
# 4. Missing duration preserved
# ============================================================

missing_duration_count = (
    df_mongo_silver
    .filter(F.col("duration_sec").isNull())
    .count()
)

flagged_missing_duration_count = (
    df_mongo_silver
    .filter(F.col("is_missing_duration") == True)
    .count()
)

assert missing_duration_count == flagged_missing_duration_count

print("PASS — Missing durations preserved and flagged.")


# ============================================================
# 5. Invalid duration preserved
# ============================================================

invalid_duration_count = (
    df_mongo_silver
    .filter(
        F.col("duration_sec").isNotNull()
        & (
            (F.col("duration_sec") <= 0)
            | (F.col("duration_sec") > 10800)
        )
    )
    .count()
)

flagged_invalid_duration_count = (
    df_mongo_silver
    .filter(F.col("is_invalid_duration") == True)
    .count()
)

assert invalid_duration_count == flagged_invalid_duration_count

print("PASS — Invalid durations preserved and flagged.")


# ============================================================
# 6. Live videos without match_id preserved
# ============================================================

invalid_live_match_count = (
    df_mongo_silver
    .filter(
        (F.col("content_type") == "live")
        & F.col("match_id").isNull()
    )
    .count()
)

flagged_live_match_count = (
    df_mongo_silver
    .filter(
        F.col("is_live_without_match_id") == True
    )
    .count()
)

assert invalid_live_match_count == flagged_live_match_count

print("PASS — Live videos without match_id preserved and flagged.")


# ============================================================
# 7. Likes greater than views preserved
# ============================================================

invalid_likes_count = (
    df_mongo_silver
    .filter(
        F.col("stats_likes").isNotNull()
        & F.col("stats_views").isNotNull()
        & (
            F.col("stats_likes")
            > F.col("stats_views")
        )
    )
    .count()
)

flagged_likes_count = (
    df_mongo_silver
    .filter(
        F.col("is_likes_greater_than_views") == True
    )
    .count()
)

assert invalid_likes_count == flagged_likes_count

print("PASS — Invalid likes/views relationships preserved and flagged.")


# ============================================================
# 8. Invalid timestamp order preserved
# ============================================================

invalid_timestamp_count = (
    df_mongo_silver
    .filter(
        F.col("last_updated").isNotNull()
        & F.col("created_at").isNotNull()
        & (
            F.col("last_updated")
            < F.col("created_at")
        )
    )
    .count()
)

flagged_timestamp_count = (
    df_mongo_silver
    .filter(
        F.col("is_last_updated_before_created") == True
    )
    .count()
)

assert invalid_timestamp_count == flagged_timestamp_count

print("PASS — Invalid timestamp relationships preserved and flagged.")


# ============================================================
# 9. Flattened nested columns exist
# ============================================================

expected_columns = [
    "competition_id",
    "competition_name",
    "competition_teams",
    "technical_max_resolution",
    "technical_codec",
    "stats_views",
    "stats_likes",
]

for column_name in expected_columns:
    assert column_name in df_mongo_silver.columns

print("PASS — Nested MongoDB fields flattened correctly.")


# ============================================================
# 10. Analytical date columns exist
# ============================================================

assert "created_date" in df_mongo_silver.columns
assert "last_updated_date" in df_mongo_silver.columns

print("PASS — Analytical date columns created.")


# ============================================================
# FINAL RESULT
# ============================================================

print("=" * 70)
print("ALL MONGODB SILVER VALIDATIONS PASSED")
print("=" * 70)

MONGODB SILVER VALIDATION


PASS — Record count preserved.


PASS — video_id uniqueness preserved.
PASS — Missing titles preserved and flagged.
PASS — Missing durations preserved and flagged.
PASS — Invalid durations preserved and flagged.
PASS — Live videos without match_id preserved and flagged.
PASS — Invalid likes/views relationships preserved and flagged.
PASS — Invalid timestamp relationships preserved and flagged.
PASS — Nested MongoDB fields flattened correctly.
PASS — Analytical date columns created.
ALL MONGODB SILVER VALIDATIONS PASSED


# MongoDB Silver — Final Transformation Report

## Transformation Summary

The MongoDB Bronze video metadata was transformed into the Silver layer while preserving the original video-level grain and source business information.

The transformation focused on technical restructuring, analytical usability, and explicit data-quality visibility without silently modifying or removing imperfect source records.

## Structure Transformation

The MongoDB nested structures were flattened into analytical columns:

- `competition.id` → `competition_id`
- `competition.name` → `competition_name`
- `competition.teams` → `competition_teams`
- `technical.max_resolution` → `technical_max_resolution`
- `technical.codec` → `technical_codec`
- `stats.views` → `stats_views`
- `stats.likes` → `stats_likes`

The original array fields were preserved:

- `languages`
- `tags`
- `competition_teams`

The arrays were not exploded, so the original video-level grain was maintained.

## Identifier Handling

`video_id` was preserved as the analytical video identifier.

MongoDB's internal `_id` was not included in the Silver analytical dataset because it is a source-system technical identifier and is not part of the analytical video model.

## Data Quality Handling

Known imperfect source values were preserved rather than deleted or silently corrected.

Explicit data-quality flags were added for:

- missing titles
- missing durations
- invalid duration values
- live videos without `match_id`
- likes greater than views
- `last_updated` occurring before `created_at`

These flags expose data-quality conditions while keeping the original source information available for downstream analysis.

## Timestamp Handling

`created_at` and `last_updated` were preserved as timestamps.

Two analytical date columns were derived:

- `created_date`
- `last_updated_date`

The transformation did not alter the original timestamp values.

## Data Preservation

The transformation preserved the original number of records.

The validation stage confirmed:

- record count preservation
- `video_id` uniqueness
- preservation of missing values
- preservation of invalid values
- correct data-quality flags
- correct flattening of nested fields
- preservation of analytical arrays
- correct creation of analytical date columns

## Final Conclusion

The MongoDB Bronze dataset has been successfully transformed into the Silver layer.

The resulting Silver dataset provides a flatter and more analytically accessible structure while preserving the original video grain, identifiers, arrays, timestamps, and imperfect source values.

Data-quality issues are made explicit through validation flags rather than being silently removed or corrected.

The MongoDB Silver transformation passed the independent validation checks and is ready for downstream Silver processing.

In [7]:
df_mongo_silver.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- match_id: string (nullable = true)
 |-- content_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- duration_sec: integer (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- competition_id: string (nullable = true)
 |-- competition_name: string (nullable = true)
 |-- competition_teams: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- technical_max_resolution: string (nullable = true)
 |-- technical_codec: string (nullable = true)
 |-- stats_views: integer (nullable = true)
 |-- stats_likes: integer (nullable = true)
 |-- languages: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- is_missing_title: boolean (nullable = true)
 |-- is_missing

In [4]:
gold_dim_video = (
    df_mongo_silver
    .filter(
        (F.col("is_missing_title") == False) &
        (F.col("is_missing_duration") == False) &
        (F.col("is_invalid_duration") == False) &
        (F.col("is_live_without_match_id") == False) &
        (F.col("is_likes_greater_than_views") == False) &
        (F.col("is_last_updated_before_created") == False)
    )
    .select(
        F.col("video_id"),
        F.col("match_id"),
        F.col("content_type"),
        F.col("status"),
        F.col("title"),
        F.col("duration_sec"),
        F.col("competition_name"),
        F.col("technical_max_resolution"),
        F.col("stats_views"),
        F.col("stats_likes"),        
        F.col("languages").getItem(0).alias("language_1"), 
        F.col("languages").getItem(1).alias("language_2"),  
        F.col("created_date").alias("created_at"),
        F.col("last_updated_date").alias("last_updated")
    )
)

In [5]:
gold_dim_video \
    .write \
    .mode("overwrite") \
    .parquet("s3a://end-to-end-streaming-data-platform-gold/dim_videos/")

In [6]:
df_dim_video = spark.read.parquet(
    "s3a://end-to-end-streaming-data-platform-gold/dim_videos/"
)

In [7]:
df_dim_video.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- match_id: string (nullable = true)
 |-- content_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- title: string (nullable = true)
 |-- duration_sec: integer (nullable = true)
 |-- competition_name: string (nullable = true)
 |-- technical_max_resolution: string (nullable = true)
 |-- stats_views: integer (nullable = true)
 |-- stats_likes: integer (nullable = true)
 |-- language_1: string (nullable = true)
 |-- language_2: string (nullable = true)
 |-- created_at: date (nullable = true)
 |-- last_updated: date (nullable = true)

